# Administration CSV generator

Turns a GeoJSON of administrative boundaries into the CSV that
`administration_csv_seeder` imports.

```
GeoJSON -> this notebook -> storage/administrations/*.csv -> seeder -> workspace
```

Everything is driven by `config.json`. See `README.md` for the field
reference; run the cells in order.

In [ ]:
import csv
import json
import re
from collections import Counter, defaultdict
from pathlib import Path


def find_repo_root(start=None):
    """Walk up to the directory holding dc.sh.

    Config paths are repo-root-relative so the same `output` value works
    whether the notebook is launched from its own directory or from the
    repo root.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "dc.sh").is_file():
            return candidate
    raise RuntimeError("could not locate the repo root (no dc.sh found)")


ROOT = find_repo_root()
HERE = ROOT / "scripts" / "administration_csv_generator"

config_path = HERE / "config.json"
if not config_path.is_file():
    config_path = HERE / "config.json.example"
    print(f"NOTE: no config.json; falling back to {config_path.name}.")
    print("      Copy it to config.json and edit before generating.\n")

CONFIG = json.loads(config_path.read_text())

# Each path gets a default directory, so the common case is a bare
# filename: the GeoJSONs live in one place and the CSVs have to land in
# storage/ for the backend container to read them. A value containing a
# separator escapes that and is taken relative to the repo root, and an
# absolute path is used as-is.
GEOJSON_DIR = HERE / "geojson"
STORAGE_DIR = ROOT / "storage" / "administrations"


def resolve_path(value, default_dir):
    raw = str(value).strip()
    if "/" in raw or "\\" in raw:
        return ROOT / raw
    return default_dir / raw


INPUT_GEOJSON = resolve_path(CONFIG["input"], GEOJSON_DIR)
OUTPUT_CSV = resolve_path(CONFIG["output"], STORAGE_DIR)
OPTIONS = CONFIG.get("options") or {}
NA_VALUES = {str(v) for v in OPTIONS.get("na_values", ["NA", ""])}
SPLIT_CAMEL_CASE = bool(OPTIONS.get("split_camel_case", False))

print("config:", config_path)
print("input :", INPUT_GEOJSON, "(exists)" if INPUT_GEOJSON.is_file() else "(MISSING)")
print("output:", OUTPUT_CSV)

## Step 1 - Preview the properties

Load the file and look at one feature. Every mapping decision is made from
this list, so run this before editing `config.json`.

In [ ]:
with INPUT_GEOJSON.open() as handle:
    geojson = json.load(handle)

features = geojson.get("features", [])
print(f"{len(features)} features\n")

sample = features[0]["properties"]
width = max(len(k) for k in sample)
print("-- properties of feature[0] --")
for key, value in sample.items():
    print(f"  {key:<{width}} = {value!r}")

In [ ]:
# A property present on only some features cannot drive a level: the rows
# missing it produce blank names, which the seeder rejects as a hole in
# the path.
present = Counter()
for feature in features:
    present.update(feature["properties"].keys())

print("-- coverage --")
for key, count in present.most_common():
    flag = "" if count == len(features) else "   <-- PARTIAL"
    print(f"  {key:<12} {count}/{len(features)}{flag}")

# GADM writes the string "NA" rather than null.
placeholder = Counter()
for feature in features:
    for key, value in feature["properties"].items():
        if isinstance(value, str) and value.strip() in NA_VALUES:
            placeholder[key] += 1
if placeholder:
    print("\n-- values matching na_values (treated as empty) --")
    for key, count in placeholder.most_common():
        print(f"  {key:<12} {count}/{len(features)}")

In [ ]:
def suggest_config(properties):
    """A starting `labels` + `properties` block, guessed from the file.

    GADM uses COUNTRY/GID_0 for the root and NAME_<n>/GID_<n> below it.
    Labels are the one thing that cannot be guessed: only the deepest tier
    carries ENGTYPE_<n>, so everything else falls back to a placeholder
    you are expected to replace.
    """
    depths = sorted(
        int(m.group(1))
        for key in properties
        for m in [re.fullmatch(r"NAME_(\d+)", key)]
        if m
    )
    labels, props = {}, {}
    if "COUNTRY" in properties:
        labels["0"] = "National"
        props["0_name"] = "COUNTRY"
        if "GID_0" in properties:
            props["0_code"] = "GID_0"
    for depth in depths:
        engtype = properties.get(f"ENGTYPE_{depth}")
        labels[str(depth)] = (
            engtype
            if isinstance(engtype, str) and engtype.strip() not in NA_VALUES
            else f"Level {depth}"
        )
        props[f"{depth}_name"] = f"NAME_{depth}"
        if f"GID_{depth}" in properties:
            props[f"{depth}_code"] = f"GID_{depth}"
    return {"labels": labels, "properties": props}


print("Suggested config -- paste into config.json and fix the labels:\n")
print(json.dumps(suggest_config(sample), indent=2))

## Step 2 - Read the mapping

`labels` decides what each tier is *called*. The seeder derives
`Levels.name` from it, so `"1": "Province"` creates a level named
"Province" that people see throughout the app — it is not just a column
heading.

In [ ]:
def parse_levels(config):
    """config -> [{level, label, name_prop, code_prop}], ordered."""
    props = config.get("properties") or {}
    labels = {str(k): v for k, v in (config.get("labels") or {}).items()}

    depths = sorted({
        int(m.group(1))
        for key in props
        for m in [re.fullmatch(r"(\d+)_(name|code)", key)]
        if m
    })
    levels = []
    for depth in depths:
        name_prop = props.get(f"{depth}_name")
        if not name_prop:
            raise ValueError(f"config.properties is missing '{depth}_name'")
        label = labels.get(str(depth))
        if not label:
            label = f"Level {depth}"
            print(
                f"WARNING: no labels['{depth}']; using {label!r}. "
                "The workspace will show that as the tier's name."
            )
        levels.append({
            "level": depth,
            "label": str(label).strip(),
            "name_prop": name_prop,
            "code_prop": props.get(f"{depth}_code"),
        })
    return levels


LEVELS = parse_levels(CONFIG)
for entry in LEVELS:
    code_prop = entry["code_prop"] or "-"
    print(
        f"  level {entry['level']}  {entry['label']:<16} "
        f"name={entry['name_prop']:<10} code={code_prop}"
    )

In [ ]:
_CAMEL = [
    (re.compile(r"(?<=[a-z])(?=[A-Z])"), " "),      # AcehBarat -> Aceh Barat
    (re.compile(r"(?<=[A-Z])(?=[A-Z][a-z])"), " "),  # DKIJakarta -> DKI Jakarta
]


def clean(value):
    """Property value -> cell value. Empty string means 'no value'."""
    if value is None:
        return ""
    text = str(value).strip()
    if text in NA_VALUES:
        return ""
    if SPLIT_CAMEL_CASE:
        for pattern, replacement in _CAMEL:
            text = pattern.sub(replacement, text)
    return text


print(f"split_camel_case = {SPLIT_CAMEL_CASE}\n")
print("-- how names will be written --")
for feature in features[:8]:
    props = feature["properties"]
    print("  " + " / ".join(
        clean(props.get(e["name_prop"])) for e in LEVELS[1:]
    ))

## Step 3 - Validate

Every check here mirrors a rule the seeder enforces, so a clean run means
the import will succeed.

In [ ]:
def validate(features, levels):
    problems = []

    depths = [e["level"] for e in levels]
    if depths != list(range(len(depths))):
        problems.append(f"levels must be contiguous from 0; got {depths}")

    labels = [e["label"] for e in levels]
    for label in labels:
        if label.lower() == "code":
            problems.append(
                f"label {label!r} collides with the '<level>_Code' column"
            )
    if len(set(labels)) != len(labels):
        problems.append(f"labels must be unique; got {labels}")

    keys = set()
    for feature in features:
        keys.update(feature["properties"].keys())
    for entry in levels:
        for role in ("name_prop", "code_prop"):
            prop = entry[role]
            if prop and prop not in keys:
                problems.append(
                    f"level {entry['level']}: {role} {prop!r} is not in "
                    "the file"
                )

    # A blank tier with a non-blank descendant is a hole in the path.
    # seed_administrations walks parent -> child and cannot bridge one.
    holes = 0
    for index, feature in enumerate(features):
        props = feature["properties"]
        blank_at = None
        for entry in levels:
            value = clean(props.get(entry["name_prop"]))
            if not value:
                if blank_at is None:
                    blank_at = entry
                continue
            if blank_at is not None:
                holes += 1
                if holes <= 3:
                    problems.append(
                        f"feature[{index}]: {blank_at['name_prop']} is "
                        f"blank but {entry['name_prop']} is not -- a path "
                        "cannot skip a tier"
                    )
                break
    if holes > 3:
        problems.append(f"...and {holes - 3} more features with holes")

    return problems


issues = validate(features, LEVELS)
if issues:
    print("PROBLEMS -- fix config.json before continuing:\n")
    for issue in issues:
        print("  -", issue)
else:
    print("Mapping is valid.")

## Step 4 - A bounding box per unit

Everything above reads `properties` only. This step is the one that opens
`geometry`, and it produces the column that lets the fake data seeder put a
datapoint's pin inside the unit the datapoint names.

The box is taken from each feature's **largest ring**, not from all of its
rings, and largest by *area* rather than by vertex count. Both choices are
measured — see `README.md`.

In [ ]:
BBOX_COLUMN = "attr_Bounding Box"


def iter_rings(geometry):
    """Every linear ring of a Polygon or MultiPolygon."""
    kind = (geometry or {}).get("type")
    coords = (geometry or {}).get("coordinates") or []
    if kind == "Polygon":
        return list(coords)
    if kind == "MultiPolygon":
        return [ring for polygon in coords for ring in polygon]
    return []


def ring_area(ring):
    """Shoelace area in square degrees. Sign discarded."""
    total = 0.0
    for i in range(len(ring) - 1):
        total += ring[i][0] * ring[i + 1][1] - ring[i + 1][0] * ring[i][1]
    return abs(total) / 2


def unit_bbox(geometry):
    """'minLng,minLat,maxLng,maxLat' for the feature's largest ring.

    A ring is a closed loop on one side of the antimeridian, so a unit that
    straddles 180 gets a real box instead of one spanning the globe. Fiji's
    Lau and Cakaudrove provinces measure 359.9 and 360.0 degrees of longitude
    over all their rings; over their largest ring, 0.13 and 1.04.

    Largest by area, not by vertex count: vertex count measures how finely a
    coastline was digitised, not how big the island is, so a heavily surveyed
    islet can outvote the mainland.
    """
    rings = iter_rings(geometry)
    if not rings:
        return ""
    ring = max(rings, key=ring_area)
    xs = [point[0] for point in ring]
    ys = [point[1] for point in ring]
    return ",".join(
        f"{value:.6g}" for value in (min(xs), min(ys), max(xs), max(ys))
    )


boxes = [unit_bbox(feature.get("geometry")) for feature in features]
missing = sum(1 for box in boxes if not box)
print(f"{len(boxes) - missing}/{len(boxes)} features have a bounding box")
if missing:
    print(f"\nWARNING: {missing} features have no usable geometry. Their rows")
    print("         get an empty box, and fake_complete_data_seeder will not")
    print("         attach datapoints to those units.")

leaf_prop = LEVELS[-1]["name_prop"]
print("\n-- sample --")
for feature, box in list(zip(features, boxes))[:5]:
    name = clean(feature["properties"].get(leaf_prop))
    print(f"  {name:<28} {box}")

## Step 5 - Build the rows

One row per deepest unit, deduplicated on the full path, so a GeoJSON
carrying several polygons for the same unit contributes one row.

In [ ]:
header = []
for entry in LEVELS:
    header.append(f"{entry['level']}_{entry['label']}")
    if entry["code_prop"]:
        header.append(f"{entry['level']}_Code")
header.append(BBOX_COLUMN)

rows, seen = [], {}
for feature, box in zip(features, boxes):
    props = feature["properties"]
    row, path = [], []
    for entry in LEVELS:
        name = clean(props.get(entry["name_prop"]))
        path.append(name)
        row.append(name)
        if entry["code_prop"]:
            row.append(clean(props.get(entry["code_prop"])))
    row.append(box)
    key = tuple(path)
    area = max(
        (ring_area(ring) for ring in iter_rings(feature.get("geometry"))),
        default=0.0,
    )
    if key in seen:
        # Several features for one unit. Keep the box of the biggest ring
        # rather than whichever came first, and never union the two -- a
        # union across separate islands re-opens the antimeridian problem
        # that picking one ring exists to close.
        index, best_area = seen[key]
        if area > best_area:
            rows[index][-1] = box
            seen[key] = (index, area)
        continue
    seen[key] = (len(rows), area)
    rows.append(row)

print("header:", ",".join(header))
print(
    f"\n{len(rows)} rows from {len(features)} features "
    f"({len(features) - len(rows)} duplicate paths collapsed)\n"
)
for row in rows[:5]:
    print("  " + ",".join(row))

## Step 6 - Sanity checks

In [ ]:
name_index = {
    entry["level"]: header.index(f"{entry['level']}_{entry['label']}")
    for entry in LEVELS
}

print("-- distinct units per tier --")
for entry in LEVELS:
    column = f"{entry['level']}_{entry['label']}"
    values = {r[name_index[entry["level"]]] for r in rows}
    values.discard("")
    print(f"  level {entry['level']} {column:<20} {len(values)}")

# The seeder requires exactly one level-0 value: a workspace has one root
# (unique_root_administration_per_tenant).
roots = {r[name_index[LEVELS[0]["level"]]] for r in rows}
print(f"\n-- root values: {sorted(roots)}")
if len(roots) != 1:
    print("  PROBLEM: the seeder requires exactly one.")

# Why the seeder keys on (name, level, parent, tenant) rather than on name
# alone: these would silently collapse under a name-only upsert.
if len(LEVELS) >= 2:
    leaf = LEVELS[-1]["level"]
    parents = defaultdict(set)
    for row in rows:
        parents[row[name_index[leaf]]].add(
            tuple(row[name_index[e["level"]]] for e in LEVELS[:-1])
        )
    shared = {n: p for n, p in parents.items() if len(p) > 1}
    print(f"\n-- {len(shared)} leaf names appear under more than one parent")
    for name, paths in sorted(shared.items(), key=lambda kv: -len(kv[1]))[:3]:
        print(f"     {name!r} under {len(paths)} parents")

# Case-insensitive sibling collisions DO merge: seed_administrations
# matches on name__iexact within a parent, so "Kota Bogor" and "KOTA BOGOR"
# under the same parent become one unit.
merges = 0
folded = defaultdict(set)
for row in rows:
    parent = None
    for entry in LEVELS:
        name = row[name_index[entry["level"]]]
        if not name:
            break
        folded[(entry["level"], name.lower(), parent)].add(name)
        parent = (entry["level"], name.lower(), parent)
for key, variants in folded.items():
    if len(variants) > 1:
        merges += 1
        if merges <= 3:
            print(f"\n  MERGE: level {key[0]} {sorted(variants)} "
                  "differ only by case and will become one unit")
if merges:
    print(f"\n  {merges} case-insensitive collisions total.")

## Step 7 - Write the CSV

In [ ]:
if issues:
    raise SystemExit("Step 3 reported problems; fix config.json first.")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(header)
    writer.writerows(rows)

print(f"wrote {len(rows)} rows to {OUTPUT_CSV} "
      f"({OUTPUT_CSV.stat().st_size:,} bytes)")
print("\n-- first 3 lines --")
with OUTPUT_CSV.open() as handle:
    for _ in range(3):
        print("  " + handle.readline().rstrip())

# The seeder takes a path relative to STORAGE_PATH, which is bind-mounted
# from the repo's storage/ directory.
try:
    relative = OUTPUT_CSV.relative_to(ROOT / "storage")
    print(f"\n--source {relative}")
except ValueError:
    print("\nNOTE: output is outside storage/, so the backend container "
          "cannot read it. Point `output` at storage/administrations/.")

## Step 8 - How good are these boxes?

A bounding box is not a polygon, and it never will be. This step measures the
gap for *this* file rather than quoting someone else's number, using the
geometry already loaded: it draws points from each stored box and ray-casts
them against the real polygons.

Two numbers matter. **Inside own unit** is how often a pin lands in the unit
its datapoint names. **On land** is how often it lands anywhere real — a pin
in a neighbouring district still reads as a plausible map; a pin at sea does
not.

For reference: a contiguous landmass scores around 51% / 96%, and a
fragmented archipelago around 44% / 70%. A low number here is a property of
the country's shape, not a bug — but it explains the map before someone
files it as one.

In [ ]:
import random

random.seed(0)
SAMPLES_PER_UNIT = 10
MAX_UNITS = 300


def in_ring(x, y, ring):
    """Ray casting. True when (x, y) is inside this ring."""
    inside = False
    count = len(ring)
    j = count - 1
    for i in range(count):
        xi, yi = ring[i][0], ring[i][1]
        xj, yj = ring[j][0], ring[j][1]
        if ((yi > y) != (yj > y)) and (
            x < (xj - xi) * (y - yi) / ((yj - yi) or 1e-15) + xi
        ):
            inside = not inside
        j = i
    return inside


# Testing every polygon per point is too slow, so each feature's own extent
# is a cheap pre-filter and only the survivors are tested properly.
indexed = []
for feature in features:
    rings = iter_rings(feature.get("geometry"))
    if not rings:
        continue
    xs = [p[0] for ring in rings for p in ring]
    ys = [p[1] for ring in rings for p in ring]
    indexed.append((min(xs), min(ys), max(xs), max(ys), rings))


def on_land(x, y):
    for x0, y0, x1, y1, rings in indexed:
        if x0 <= x <= x1 and y0 <= y <= y1:
            if any(in_ring(x, y, ring) for ring in rings):
                return True
    return False


population = [
    (feature, box) for feature, box in zip(features, boxes) if box
]
sample = (
    random.sample(population, MAX_UNITS)
    if len(population) > MAX_UNITS else population
)

own = land = total = 0
worst = []
for feature, box in sample:
    min_lng, min_lat, max_lng, max_lat = [float(v) for v in box.split(",")]
    rings = iter_rings(feature.get("geometry"))
    hits = 0
    for _ in range(SAMPLES_PER_UNIT):
        x = random.uniform(min_lng, max_lng)
        y = random.uniform(min_lat, max_lat)
        total += 1
        if any(in_ring(x, y, ring) for ring in rings):
            own += 1
            land += 1
            hits += 1
        elif on_land(x, y):
            land += 1
    worst.append((hits / SAMPLES_PER_UNIT,
                  clean(feature["properties"].get(leaf_prop))))

print(f"{total} points across {len(sample)} units:")
print(f"  inside own unit  {own / total:5.0%}")
print(f"  on land anywhere {land / total:5.0%}")

print("\n-- units where pins land outside most often --")
for rate, name in sorted(worst)[:5]:
    print(f"  {name:<28} {rate:4.0%} inside")

widest = max(
    (float(b.split(",")[2]) - float(b.split(",")[0]), b) for _f, b in population
)
print(f"\nwidest stored box spans {widest[0]:.2f} deg of longitude")
if widest[0] > 180:
    print("  PROBLEM: a stored box crosses the antimeridian. Picking one ring")
    print("  should make that impossible -- check unit_bbox above.")

## Step 9 - Import into a workspace

Dry-run first: it validates the whole file and rolls back.

```bash
./dc.sh exec backend python manage.py administration_csv_seeder \
    --source administrations/indonesia.csv --tenant <subdomain> --dry-run

./dc.sh exec backend python manage.py administration_csv_seeder \
    --source administrations/indonesia.csv --tenant <subdomain>
```

If the workspace already has a root under a different name, the seeder
stops and names both rather than guessing; pass `--rename-root` to accept
the file's value.